RETRIEVER

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader,get_response_synthesizer
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Settings

Settings.chunk_size = 128
Settings.chunk_overlap = 50

documents = SimpleDirectoryReader("/home/jjh_test/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(verbose=True, similarity_top_k=2)
response_synthesizer = get_response_synthesizer(
    response_mode="compact",
)
query_engine = index.as_query_engine()

In [2]:
eval_q_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-q").load_data()

In [3]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [4]:
retriever_result = {}
search_results = []

for count, question in enumerate(questions, start=1):
    search_this = f"{question}"
    search_result = retriever.retrieve(search_this)
    search_results = []
    
    for i in range(retriever.similarity_top_k):
        search_results.append(search_result[i].get_text())

    retriever_result[count] = search_results

EMBEDDING MODEL

In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

GENERATOR

In [6]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)

In [7]:
import transformers
import torch

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

KeyError: 'type'

In [ ]:
responses=[]

In [ ]:
for count, question in enumerate(questions, start=1):
    retriever_context_list = retriever_result.get(count, [])
    retriever_context = ' '.join(retriever_context_list)
    
    query = f" Given the context: '{retriever_context}', determine if the following statement is 'True' or 'False': {question}. Start your response with 'True' or 'False'. Answer:"
    
    messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": query},
    ]
    
    outputs = pipeline(
        messages,
        max_new_tokens=256,
    )
        
    aa = response.find("Answer:")
    if aa != -1:
        response = response[aa + 7:aa + 14] 
    else:
        response = response[:6]  

    responses.append(response)

    print(count, response)

EVALUATOR

In [ ]:
eval_a_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-a").load_data()

In [ ]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [ ]:
responses_str = []

In [ ]:
for response in responses:
    response_str = str(response)
    if "True" in response_str:
        response_str = "True"
    elif "False" in response_str:
        response_str = "False"
    
    responses_str.append(response_str)

In [ ]:
correct_count=0
number=0

for answer, response, response_str, question in zip(answers, responses, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

In [ ]:
accuracy = (correct_count / len(questions)) * 100
print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")